# `microstructure_demo` — Iterative Improvement Loop

Iteratively raises the CV R² on Cycle 1 HoldingTemp / HoldingTime by stacking the strategies discussed in the analysis (log-transformed time, per-target single-output models, stratified KFolds). Caches are read **read-only** from the demo notebook's `data/image_cache_*.npz` and `features/morph_features_c1.npz` — nothing is rebuilt.

All real work lives in [`scripts/iterate_demo.py`](../scripts/iterate_demo.py). Each iteration writes:

- `runs/iterate_demo/<run_id>/iteration_<n>.md` — what was tried, code diff, deltas, winner
- `runs/iterate_demo/<run_id>/summary.md` — running cumulative table
- `runs/iterate_demo/<run_id>/history.csv` — long-format row per (iteration, strategy, target)

Ordering of Tier-1 strategies (tried left-to-right per iteration; the largest positive Δ-mean-R² wins and is folded into the baseline):

1. `log_time` — wrap HoldingTime with `TransformedTargetRegressor(log1p)` so linear MSE doesn't over-weight long-time samples.
2. `per_target` — train two single-output GBRs instead of `MultiOutputRegressor` wrapping a joint GBR.
3. `stratify_alloy` — `StratifiedKFold` by alloy so every alloy is represented in train and test.
4. `stratify_temp_bin` — `StratifiedKFold` by HoldingTemp quantile-bin so rare setpoints aren't isolated.

Stop criterion: no remaining strategy clears `--min-delta` (default `0.005`) on the mean of (HoldingTemp R², HoldingTime R²).

In [ ]:
import subprocess, sys, os
from pathlib import Path

REPO = Path(os.path.abspath('..'))
script = REPO / 'scripts' / 'iterate_demo.py'
print(f'Running: {script}')

result = subprocess.run(
    [sys.executable, str(script),
     '--csv',       str(REPO / 'datasets' / 'metadata_latest.csv'),
     '--backbone',  'dinov2_vitb14',
     '--cache-dir', str(REPO / 'data'),
     '--morph-cache', str(REPO / 'features' / 'morph_features_c1.npz'),
     '--tier',      '1',
     '--min-delta', '0.005',
     '--cv-repeats', '3',
     '--final-cv-repeats', '10'],
    cwd=str(REPO),
)
print(f'\nExit code: {result.returncode}')

In [ ]:
# Show the latest run's summary inline.
from pathlib import Path
import os
REPO = Path(os.path.abspath('..'))
run_root = REPO / 'runs' / 'iterate_demo'
latest = None
if run_root.exists():
    runs = sorted([p for p in run_root.iterdir() if p.is_dir()])
    if runs:
        latest = runs[-1]
        print(f'Latest run: {latest}')
        summary = latest / 'summary.md'
        if summary.exists():
            print()
            print(summary.read_text())

In [ ]:
# Show the per-iteration markdown logs.
if latest is not None:
    for it_md in sorted(latest.glob('iteration_*.md')):
        print(f"\n{'=' * 70}")
        print(f'  {it_md.name}')
        print(f"{'=' * 70}\n")
        print(it_md.read_text())